# Descarga del Dataset

In [1]:
from huggingface_hub import snapshot_download

ruta = snapshot_download(
    repo_id="phamquiluan/RCAEval",
    repo_type="dataset",
    allow_patterns="re2ob_*/*",
    local_dir="data/RCAEval",
)

print(ruta)

/home/lambda/Obsidian/Segundo Cerebro/1.Proyectos/2026-2/Fundamentos de Inteligencia artificial/RandomForest-Implementation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 360 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 360/360 [04:36<00:00,  1.30it/s]

/home/lambda/Obsidian/Segundo Cerebro/1.Proyectos/2026-2/Fundamentos de Inteligencia artificial/RandomForest-Implementation/data/RCAEval


# Cargar el dataset y mostrar datos

In [3]:

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Configuración
BASE = Path("data/RCAEval")
if not (BASE / "cases.parquet").exists():
    BASE = Path("../data/RCAEval")

IDX = BASE / "cases.parquet"

if not IDX.exists():
    raise FileNotFoundError(
        f"No se encontró cases.parquet en {BASE}"
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)


def titulo(txt):
    print("\n" + "=" * 75)
    print(txt)
    print("=" * 75)


def resumen(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "nulos_%": (
            df.isna().mean() * 100
        ).round(2),
        "unicos": df.nunique(dropna=True),
    })


def muestra(df, nombre, n=5):
    titulo(nombre)
    print("Dimensiones:", df.shape)
    print("Memoria (MiB):", round(
        df.memory_usage(deep=True).sum()
        / 1024**2, 2
    ))
    display(resumen(df))
    display(df.head(n))


# 1. Índice del dataset
idx = pd.read_parquet(IDX)
re2 = idx.loc[
    idx["dataset"] == "RE2-OB"
].copy()

titulo("1. INFORMACIÓN GENERAL")
print("Casos del subconjunto:", len(re2))
print("Columnas del índice:", len(re2.columns))
display(re2.head())
display(resumen(re2))

# 2. Etiquetas y distribución
titulo("2. TIPOS DE FALLAS")
display(
    re2["fault"].value_counts()
    .rename_axis("falla")
    .reset_index(name="casos")
)

titulo("3. SERVICIOS CAUSANTES")
display(
    re2["root_cause_service"]
    .value_counts()
    .rename_axis("servicio")
    .reset_index(name="casos")
)

titulo("4. FALLAS POR SERVICIO")
display(pd.crosstab(
    re2["root_cause_service"],
    re2["fault"]
))

# 3. Cobertura general
titulo("5. COBERTURA DEL DATASET")
cols = [
    "n_metrics", "n_timesteps",
    "duration_minutes",
    "normal_timesteps",
    "faulty_timesteps",
    "n_logs", "n_traces"
]
cols = [c for c in cols if c in re2.columns]

display(re2[cols].describe().T)

# 4. Archivos disponibles
titulo("6. ARCHIVOS DESCARGADOS")

filas = []
for _, fila in re2.iterrows():
    carp = BASE / fila["case"]
    reg = {"caso": fila["case"]}
    for nombre in [
        "metrics.parquet",
        "logs.parquet",
        "traces.parquet",
        "inject_time.txt"
    ]:
        arch = carp / nombre
        reg[nombre] = arch.exists()
        reg[nombre + "_MiB"] = (
            round(arch.stat().st_size / 1024**2, 2)
            if arch.exists() else np.nan
        )
    filas.append(reg)

archivos = pd.DataFrame(filas)
display(archivos.head(10))

print("Disponibilidad por archivo:")
display(
    archivos[
        [c for c in archivos.columns
         if c.endswith((".parquet", ".txt"))]
    ].sum()
)

# 5. Seleccionar un caso disponible
validos = [
    c for c in re2["case"]
    if (BASE / c / "metrics.parquet").exists()
]

if not validos:
    raise FileNotFoundError(
        "No hay métricas RE2-OB descargadas."
    )

caso = validos[0]
carp = BASE / caso
meta = re2.loc[re2["case"] == caso].iloc[0]

titulo("7. CASO SELECCIONADO")
print("Caso:", caso)
print("Servicio raíz:", meta["root_cause_service"])
print("Tipo de falla:", meta["fault"])
print("Repetición:", meta["repetition"])
print("Inyección Unix:", meta["inject_time"])
print("Inyección UTC:", pd.to_datetime(
    meta["inject_time"], unit="s", utc=True
))

# 6. Métricas
met = pd.read_parquet(carp / "metrics.parquet")
muestra(met, "8. ESTRUCTURA DE MÉTRICAS")

titulo("9. ESTADÍSTICAS DE MÉTRICAS")
num = met.select_dtypes(include=np.number)
num = num.drop(columns=["time"], errors="ignore")

display(num.describe().T)

titulo("10. MÉTRICAS CONSTANTES")
const = num.columns[
    num.nunique(dropna=True) <= 1
].tolist()
print("Cantidad:", len(const))
print(const)

titulo("11. MÉTRICAS CON MÁS NULOS")
display(
    resumen(num)
    .sort_values("nulos_%", ascending=False)
    .head(20)
)

# 7. Temporalidad
titulo("12. ANÁLISIS TEMPORAL")

if "time" in met.columns:
    t = pd.to_numeric(
        met["time"], errors="coerce"
    ).dropna().sort_values().drop_duplicates()

    print("Inicio UTC:", pd.to_datetime(
        t.min(), unit="s", utc=True
    ))
    print("Fin UTC:", pd.to_datetime(
        t.max(), unit="s", utc=True
    ))

    dt = t.diff().dropna()
    print("Intervalos (segundos):")
    display(dt.describe())

    iny = int(meta["inject_time"])
    normal = met[met["time"] < iny]
    falla = met[met["time"] >= iny]

    print("Filas normales:", len(normal))
    print("Filas posteriores:", len(falla))

# 8. Comparación antes/después
titulo("13. CAMBIOS EN MÉTRICAS")

if len(normal) and len(falla):
    a = normal[num.columns].mean()
    b = falla[num.columns].mean()

    comp = pd.DataFrame({
        "media_normal": a,
        "media_posterior": b,
        "diferencia": b - a,
    })
    comp["cambio_abs"] = (
        comp["diferencia"].abs()
    )

    display(
        comp.sort_values(
            "cambio_abs", ascending=False
        ).head(20)
    )

# 9. Logs y trazas
for nombre in ["logs", "traces"]:
    arch = carp / f"{nombre}.parquet"

    if arch.exists():
        df = pd.read_parquet(arch)
        muestra(
            df,
            f"14. ESTRUCTURA DE {nombre.upper()}"
        )

        if nombre == "traces":
            for col in ["serviceName", "statusCode"]:
                if col in df.columns:
                    print(f"Distribución de {col}:")
                    display(
                        df[col].value_counts().head(20)
                    )
    else:
        print(f"No existe {nombre}.parquet")

# 10. Compatibilidad entre casos
titulo("15. CONSISTENCIA DE CARACTERÍSTICAS")

esquemas = {}
for c in validos:
    ruta = BASE / c / "metrics.parquet"
    columnas = pd.read_parquet(
        ruta, columns=None
    ).columns
    esquemas[c] = set(columnas) - {"time"}

union = set().union(*esquemas.values())
inter = set.intersection(*esquemas.values())

print("Casos inspeccionados:", len(esquemas))
print("Métricas distintas:", len(union))
print("Métricas comunes:", len(inter))
print("Métricas no comunes:", len(union - inter))
print("Ejemplos de métricas comunes:")
print(sorted(inter)[:30])

titulo("EXPLORACIÓN FINALIZADA")



1. INFORMACIÓN GENERAL
Casos del subconjunto: 90
Columnas del índice: 22


,case,dataset,suite,system,system_name,root_cause_service,fault,fault_description,repetition,inject_time,n_metrics,n_timesteps,time_start,time_end,duration_minutes,normal_timesteps,faulty_timesteps,has_logs,n_logs,has_traces,n_traces,has_root_cause_file
375,re2ob_checkoutservice_cpu_1,RE2-OB,RE2,ob,Online Boutique,checkoutservice,cpu,CPU stress,1,1705354566,72,1441,1705353846,1705355286,24.0,720,721,True,171322,True,391997,False
376,re2ob_checkoutservice_cpu_2,RE2-OB,RE2,ob,Online Boutique,checkoutservice,cpu,CPU stress,2,1708475361,74,930,1708474641,1708475570,15.5,720,210,True,110067,True,199960,False
377,re2ob_checkoutservice_cpu_3,RE2-OB,RE2,ob,Online Boutique,checkoutservice,cpu,CPU stress,3,1705623539,74,1441,1705622819,1705624259,24.0,720,721,True,178461,True,408211,False
378,re2ob_checkoutservice_delay_1,RE2-OB,RE2,ob,Online Boutique,checkoutservice,delay,network delay,1,1705666511,75,1441,1705665791,1705667231,24.0,720,721,True,177868,True,405229,False
379,re2ob_checkoutservice_delay_2,RE2-OB,RE2,ob,Online Boutique,checkoutservice,delay,network delay,2,1705668921,75,1441,1705668201,1705669641,24.0,720,721,True,175563,True,402454,False


,tipo,nulos,nulos_%,unicos
case,str,0,0.0,90
dataset,str,0,0.0,1
suite,str,0,0.0,1
system,str,0,0.0,1
system_name,str,0,0.0,1
root_cause_service,str,0,0.0,5
fault,str,0,0.0,6
fault_description,str,0,0.0,6
repetition,int64,0,0.0,3
inject_time,int64,0,0.0,90



2. TIPOS DE FALLAS


,falla,casos
0,cpu,15
1,delay,15
2,disk,15
3,loss,15
4,mem,15
5,socket,15



3. SERVICIOS CAUSANTES


... (27 outputs left out)

serviceName
frontendservice          199519
productcatalogservice     99572
currencyservice           55032
recommendationservice     26434
checkoutservice            9238
emailservice               1245
paymentservice              957
Name: count, dtype: int64[pyarrow]

Distribución de statusCode:


statusCode
0    373623
Name: count, dtype: Int64


15. CONSISTENCIA DE CARACTERÍSTICAS
Casos inspeccionados: 90
Métricas distintas: 86
Métricas comunes: 67
Métricas no comunes: 19
Ejemplos de métricas comunes:
['adservice_cpu', 'adservice_diskio', 'adservice_latency-50', 'adservice_latency-90', 'adservice_mem', 'adservice_socket', 'adservice_workload', 'cartservice_cpu', 'cartservice_latency-50', 'cartservice_latency-90', 'cartservice_mem', 'cartservice_socket', 'cartservice_workload', 'checkoutservice_cpu', 'checkoutservice_latency-50', 'checkoutservice_latency-90', 'checkoutservice_mem', 'checkoutservice_socket', 'checkoutservice_workload', 'currencyservice_cpu', 'currencyservice_latency-50', 'currencyservice_latency-90', 'currencyservice_mem', 'currencyservice_socket', 'currencyservice_workload', 'emailservice_cpu', 'emailservice_diskio', 'emailservice_latency-50', 'emailservice_latency-90', 'emailservice_mem']

EXPLORACIÓN FINALIZADA
